In [1]:
import os
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from tqdm import tqdm
import random 
import numpy as np

# Load CSV
df = pd.read_csv("Data_Entry_2017_v2020.csv")

df['Finding Labels'] = df['Finding Labels'].str.split('|')
desiredClasses = ['No Finding', 'Pneumothorax', 'Cardiomegaly', 'Consolidation', 'Nodule']
# desiredClasses = ['Pneumothorax', 'Cardiomegaly', 'Mass', 'Nodule']
desiredSet = set(desiredClasses)

df = df[df['Finding Labels'].apply(lambda x: any(label in desiredClasses for label in x))] #Gets lines where only one category in the list is present
df['Finding Labels'] = df['Finding Labels'].apply(lambda x: list(set(x) & desiredSet)) #Makes the ´Finding Labels´ column only have categories in desiredClasses

for label in desiredClasses:
    df[label] = df['Finding Labels'].apply(lambda x: int(label in x))

rng = random.Random(23062025)

df['Main Label'] = df['Finding Labels'].apply(lambda x: rng.choice(x)) #Randomly select a label to define as 'main label'

maxN = 1800
#Random sampling of classes, getting max lines possible
df = df.groupby('Main Label', group_keys=False).sample(n=maxN, random_state=23062025)
df


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Sex,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],No Finding,Pneumothorax,Cardiomegaly,Consolidation,Nodule,Main Label
59114,00014626_010.png,[Cardiomegaly],10,14626,44,F,PA,2536,2944,0.139000,0.139000,0,0,1,0,0,Cardiomegaly
88320,00021845_005.png,[Cardiomegaly],5,21845,19,M,AP,3056,2544,0.139000,0.139000,0,0,1,0,0,Cardiomegaly
16434,00004381_001.png,[Cardiomegaly],28,4381,29,M,PA,2992,2981,0.143000,0.143000,0,0,1,0,0,Cardiomegaly
53520,00013520_010.png,[Cardiomegaly],14,13520,18,M,AP,3056,2544,0.139000,0.139000,0,0,1,0,0,Cardiomegaly
103800,00027706_033.png,[Cardiomegaly],33,27706,36,M,PA,2021,2021,0.194311,0.194311,0,0,1,0,0,Cardiomegaly
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52921,00013342_013.png,[Pneumothorax],13,13342,48,M,PA,2992,2991,0.143000,0.143000,0,1,0,0,0,Pneumothorax
90916,00022651_017.png,[Pneumothorax],18,22651,20,M,PA,2624,2770,0.143000,0.143000,0,1,0,0,0,Pneumothorax
92391,00023075_007.png,[Pneumothorax],7,23075,29,M,PA,2992,2991,0.143000,0.143000,0,1,0,0,0,Pneumothorax
41826,00010790_002.png,[Pneumothorax],2,10790,55,F,AP,2500,2048,0.168000,0.168000,0,1,0,0,0,Pneumothorax


In [2]:
import tarfile
from pathlib import Path

tar_files = [
    'tars/images_001.tar.gz',
    'tars/images_002.tar.gz',
    'tars/images_003.tar.gz',
    'tars/images_004.tar.gz',
    'tars/images_005.tar.gz',
    'tars/images_006.tar.gz',
    'tars/images_007.tar.gz',
    'tars/images_008.tar.gz',
    'tars/images_009.tar.gz',
    'tars/images_010.tar.gz',
    'tars/images_011.tar.gz',
    'tars/images_012.tar.gz',
]  

# Setup output directory
output_dir = Path('./extracted_images') # final output will be /extracted_images/images
output_dir.mkdir(parents=True, exist_ok=True)

image_index_set = set(df['Image Index'])
df['Recovered_Image'] = 0

# Get only desired files directly from tar.gz so we dont blow our storage
for tar_path in tar_files:
    with tarfile.open(tar_path, 'r:gz') as tar:
        haha = tar.getmembers()
        for member in tar.getmembers():
            filename = Path(member.name).name
            filePath = output_dir / 'images' / filename
            if filePath.exists():
                df.loc[df['Image Index'] == filename, 'Recovered_Image'] = 1 #mark that image was found
                continue
            if filename in image_index_set:
                tar.extract(member, path=output_dir)
                df.loc[df['Image Index'] == filename, 'Recovered_Image'] = 1 #mark that image was found



#Drop all df entries that are not in the extracted_folder
df.drop(df[df['Recovered_Image'] == 0].index, inplace=True)
df.drop(columns=['Recovered_Image'], inplace=True)
df

C:\Users\joaoe\AppData\Local\Temp\ipykernel_18368\2537478732.py:37: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=output_dir)
C:\Users\joaoe\AppData\Local\Temp\ipykernel_18368\2537478732.py:37: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=output_dir)
C:\Users\joaoe\AppData\Local\Temp\ipykernel_18368\2537478732.py:37: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=output_dir)
C:\Users\joaoe\AppData\Local\Temp\ipykernel_18368\2537478732.py:37: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject f

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Sex,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],No Finding,Pneumothorax,Cardiomegaly,Consolidation,Nodule,Main Label
59114,00014626_010.png,[Cardiomegaly],10,14626,44,F,PA,2536,2944,0.139000,0.139000,0,0,1,0,0,Cardiomegaly
88320,00021845_005.png,[Cardiomegaly],5,21845,19,M,AP,3056,2544,0.139000,0.139000,0,0,1,0,0,Cardiomegaly
16434,00004381_001.png,[Cardiomegaly],28,4381,29,M,PA,2992,2981,0.143000,0.143000,0,0,1,0,0,Cardiomegaly
53520,00013520_010.png,[Cardiomegaly],14,13520,18,M,AP,3056,2544,0.139000,0.139000,0,0,1,0,0,Cardiomegaly
103800,00027706_033.png,[Cardiomegaly],33,27706,36,M,PA,2021,2021,0.194311,0.194311,0,0,1,0,0,Cardiomegaly
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52921,00013342_013.png,[Pneumothorax],13,13342,48,M,PA,2992,2991,0.143000,0.143000,0,1,0,0,0,Pneumothorax
90916,00022651_017.png,[Pneumothorax],18,22651,20,M,PA,2624,2770,0.143000,0.143000,0,1,0,0,0,Pneumothorax
92391,00023075_007.png,[Pneumothorax],7,23075,29,M,PA,2992,2991,0.143000,0.143000,0,1,0,0,0,Pneumothorax
41826,00010790_002.png,[Pneumothorax],2,10790,55,F,AP,2500,2048,0.168000,0.168000,0,1,0,0,0,Pneumothorax


In [3]:
#Train test split, while preserving the balance of the conditions
train_patients, val_patients = train_test_split(
    df,
    test_size=0.2,
    stratify=df['Finding Labels'],
    random_state=23062025
)

train_df = df[df['Patient ID'].isin(train_patients['Patient ID'])]
val_df = df[df['Patient ID'].isin(val_patients['Patient ID'])]

print(f"Train images: {len(train_df)}, Val images: {len(val_df)}")
print(f"Train patients: {len(train_df)}, Val patients: {len(val_df)}\n")

print("Train class distribution:")
print(train_df['Finding Labels'].value_counts())

print("\nValidation class distribution:")
print(val_df['Finding Labels'].value_counts())


ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

# Starting to make the model

In [ ]:
class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, img_dir, class_names, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.class_names = class_names

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.loc[idx, 'Image Index']
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        # Get multi-hot encoded label
        labels = self.df.loc[idx, 'Finding Labels']  # should be a list
        label_vector = np.zeros(len(self.class_names), dtype=np.long)
        for label in labels:
            if label in self.class_names:
                label_vector[self.class_names.index(label)] = 1.0

        return image, torch.tensor(label_vector)
    
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], 
                         [0.229, 0.224, 0.225])
])

train_dataset = ChestXrayDataset(train_patients, 'extracted_images/images', transform=transform)
val_dataset = ChestXrayDataset(val_patients, 'extracted_images/images', transform=transform)

# train_loader = DataLoader(train_dataset, batch_size=64, num_workers=4, shuffle=True, pin_memory=True, persistent_workers=True)
# val_loader = DataLoader(val_dataset, batch_size=64, num_workers=4, pin_memory=True, persistent_workers=True)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)

In [ ]:
import torchvision.models as models
import torch.nn as nn
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

resnet = models.resnet18(pretrained=True)
for param in resnet.parameters():
    param.requires_grad = False  # optional freeze

resnet.fc = nn.Linear(resnet.fc.in_features, len(desiredClasses))  # 5 output nodes
resnet = resnet.to(device)

d:\MC906-DiagXray\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
from sklearn.metrics import multilabel_confusion_matrix

criterion = nn.BCEWithLogitsLoss() #different criterion for multilabel
optimizer = torch.optim.Adam(resnet.fc.parameters(), lr=1e-3)

train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []
num_epochs = 10
for epoch in range(num_epochs):
    print(f"Epoch {epoch} training\n")
    # --------------------
    # TRAINING
    # --------------------
    resnet.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels, _ in train_loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = resnet(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Accuracy
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    print(f"Epoch {epoch} validation\n")
    # --------------------
    # VALIDATION
    # --------------------
    resnet.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels, _ in val_loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            outputs = resnet(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss /= len(val_loader)
    val_acc = correct / total
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    # Output the metrics
    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    mcm = multilabel_confusion_matrix((preds == labels).sum(), preds)  # List of 2x2 matrices, one per class


    for i, label in enumerate(desiredClasses):
        cm = mcm[i]
        tn, fp, fn, tp = cm.ravel()

        matrix = np.array([[tp, fn], [fp, tn]])

        fig, ax = plt.subplots()
        im = ax.imshow(matrix, cmap='Blues')

        # Set axis labels
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(['Pred: 1', 'Pred: 0'])
        ax.set_yticklabels(['True: 1', 'True: 0'])

        # Annotate cells with values
        for row in range(2):
            for col in range(2):
                ax.text(col, row, str(matrix[row, col]),
                        ha='center', va='center', color='black', fontsize=12)

        ax.set_title(f"TP/TN Matrix for \"{label}\"")
        plt.tight_layout()
        plt.show()
        

Epoch 0 training

Epoch 0 validation

Epoch 1/10 | Train Loss: 1.5765, Train Acc: 0.2871 | Val Loss: 1.5253, Val Acc: 0.3183
Epoch 1 training

Epoch 1 validation

Epoch 2/10 | Train Loss: 1.4910, Train Acc: 0.3457 | Val Loss: 1.4912, Val Acc: 0.3450
Epoch 2 training

Epoch 2 validation

Epoch 3/10 | Train Loss: 1.4509, Train Acc: 0.3883 | Val Loss: 1.4819, Val Acc: 0.3528
Epoch 3 training

Epoch 3 validation

Epoch 4/10 | Train Loss: 1.4310, Train Acc: 0.4046 | Val Loss: 1.4643, Val Acc: 0.3628
Epoch 4 training

Epoch 4 validation

Epoch 5/10 | Train Loss: 1.4208, Train Acc: 0.4029 | Val Loss: 1.4640, Val Acc: 0.3667
Epoch 5 training

Epoch 5 validation

Epoch 6/10 | Train Loss: 1.4084, Train Acc: 0.4054 | Val Loss: 1.4822, Val Acc: 0.3650
Epoch 6 training

Epoch 6 validation

Epoch 7/10 | Train Loss: 1.4088, Train Acc: 0.4089 | Val Loss: 1.4581, Val Acc: 0.3689
Epoch 7 training



In [ ]:
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

all_preds = []
all_labels = []

resnet.eval()
with torch.no_grad():
    for inputs, labels, labStr in val_loader:
        inputs = inputs.to(device)
        outputs = resnet(inputs)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=train_dataset.class_names)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()